<a href="https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**This mirrors the exact flag the deck builds live**

> **Among** articles with enough impressions to judge, not brand-new, and not optimized in the
> last two months (**population**: the eligibility gate / cooldown), **look at** click-through
> rate compared with articles at a similar position (**evidence**). **If** it is far below those
> peers (**condition**), **then** send it to snippet review, with the reason written down
> (**action**).

**Two signals this rule leans on, checked in 1a/1b below, both flag-linked, as required:**
- **CTR vs. position**: This is the rule's core *evidence*.
- **Volume** (`prev_30_impressions`): We should have enough impressions to judge. It's also literally the "rank by stake" weight:
  `stake ≈ impressions × (peer_ctr − own_ctr)`.

**Population / eligibility (all gates, not weights: a row must pass all three to be scored at all):**
- `prev_30_impressions >= 500`: "enough impressions to judge".
- `content_age_days_at_decision >= 90`: "not brand-new".
- `days_since_last_update_at_decision >= 60`: "not optimized in the last two months".

**Condition ("far below peers"):** `prev_30_ctr = 0`: documented choice, I noticed that significant amount of contents have zero clicks.

**Score ("rank by stake"):**
`prev_30_impressions * (expected_ctr_for_tier - prev_30_ctr)` for eligible, flagged rows —
estimated extra clicks recoverable if the snippet were fixed. Zero for everyone else.

**Reason code:** `CTR_BELOW_POSITION_PEERS` — the deck's own exact string.

**Action label:** `snippet_fix`:  the deck's own treatment name for this exact diagnosis.

**Inputs used by the score (all `prev_30` or static, nothing from `last_30`, nothing label-derived):** `prev_30_avg_position`, `prev_30_ctr`, `prev_30_impressions`,
`content_age_days_at_decision`, `days_since_last_update_at_decision`.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import os, sys
import pandas as pd
import numpy as np
import duckdb

REPO_URL = "https://github.com/vahagngrigoryan2006/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

# Repo checkout matters here (unlike w03): section 2 writes a CSV to a repo-relative path.
import subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

con = duckdb.connect()

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Set HF_TOKEN as a Colab secret (or env var if running locally) before continuing."

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN \'{HF_TOKEN}\')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT     = f"read_parquet(\'{WAREHOUSE}/dim_content.parquet\')"
FACT_APRIL_MAY  = (
    f"read_parquet([\'{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet\', "
    f"\'{WAREHOUSE}/fact_content_daily_performance/month=2026-04/*.parquet\', "
    f"\'{WAREHOUSE}/fact_content_daily_performance/month=2026-05/*.parquet\'])"
)

# Same window as the current w03_data_contract.ipynb: as-of 2026-05-31, decision moment 2026-05-01.
features = con.sql(f"""
    WITH bounds AS (
        SELECT DATE \'2026-05-31\' AS as_of_date
    ),
    per_item AS (
        SELECT f.client_hash_id, f.content_hash_id,
               MIN(f.report_date) AS first_seen,
               SUM(CASE WHEN f.report_date >= b.as_of_date - INTERVAL 30 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS last_30_impressions,
               SUM(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS prev_30_impressions,
               SUM(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_clicks ELSE 0 END)      AS prev_30_clicks,
               AVG(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_avg_position END)       AS prev_30_avg_position
        FROM {FACT_APRIL_MAY} f, bounds b
        GROUP BY 1, 2
    )
    SELECT p.*, d.content_created_date, d.content_updated_date
    FROM per_item p
    JOIN {DIM_CONTENT} d USING (content_hash_id)
    WHERE p.first_seen <= DATE \'2026-05-31\' - INTERVAL 60 DAY   -- guard (a): full prev_30 history
      AND p.prev_30_impressions >= 100                             -- guard (b): activity floor
""").df()

decision_moment = pd.Timestamp("2026-05-01")
features["content_created_date"] = pd.to_datetime(features["content_created_date"])
features["content_updated_date"] = pd.to_datetime(features["content_updated_date"])
features["content_age_days_at_decision"] = (decision_moment - features["content_created_date"]).dt.days
features["days_since_last_update_at_decision"] = (decision_moment - features["content_updated_date"]).dt.days
features = features[features["days_since_last_update_at_decision"] > 0]   # guard (c)

# is_declining, for the signal-2 check and precision@K -- never an input to the score itself
features["impressions_pct_change"] = (
    (features["last_30_impressions"] - features["prev_30_impressions"]) / features["prev_30_impressions"]
)
features["is_declining"] = (features["impressions_pct_change"] < -0.20).astype(int)

# prev_30_ctr, percent scale
features["prev_30_ctr"] = features["prev_30_clicks"] / features["prev_30_impressions"] * 100

df = features.copy()
print(f"Working frame: {len(df):,} content items surviving all three guards.")
print(f"is_declining rate: {df['is_declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Working frame: 18,918 content items surviving all three guards.
is_declining rate: 0.543


### 1a. Signal check: CTR vs. position

Define`position_tier` this way: `top_3` ≤3, `page_1` ≤10, `striking_distance` ≤20, `page_2_3` ≤50, `deep` >50.

**Conclusion**: We can CONFIRM, the pattern is clearly there. We can see a clear, monotonic drop in mean CTR as position worsens, with every tier's `n`
comfortably large (664 the smallest one).

In [2]:
position_bins = [0, 3, 10, 20, 50, np.inf]
position_labels = ["top_3", "page_1", "striking_distance", "page_2_3", "deep"]
df["position_tier"] = pd.cut(df["prev_30_avg_position"], bins=position_bins, labels=position_labels)

signal1_table = df.groupby("position_tier", observed=True)["prev_30_ctr"].agg(["mean", "count"]).round(3)
signal1_table.columns = ["mean_ctr_pct", "n"]
print("Signal 1 -- mean prev_30 CTR (%) by position tier:")
signal1_table

Signal 1 -- mean prev_30 CTR (%) by position tier:


,mean_ctr_pct,n
position_tier,,
top_3,0.399,664
page_1,0.261,6959
striking_distance,0.203,4519
page_2_3,0.107,5752
deep,0.035,1024


### 1b. Signal check: volume (this rule's eligibility gate + score weight)

Volume tiers below line up with this rule's own `>= 500` eligibility cutoff (`thin` is exactly
the population this rule excludes). Checked against CTR, so it reads the same way as 1a: does traffic volume itself carry information, or is
it "just" a reliability gate?

**Conclusion**: CONFIRMED, mean ctr monotonically rises with impressions, with comfortable large `n`s (smallest one is 1094). This is a real, useable pattern on top of the eligibility-gate role.

In [6]:
volume_bins = [0, 500, 2000, 5000, np.inf]
volume_labels = ["thin_lt500", "moderate_500_2000", "healthy_2000_5000", "high_5000plus"]
df["volume_tier"] = pd.cut(df["prev_30_impressions"], bins=volume_bins, labels=volume_labels)

signal2_table = df.groupby("volume_tier", observed=True)["prev_30_ctr"].agg(["mean", "count"]).round(3)
signal2_table.columns = ["mean_ctr_pct", "n"]
print("Signal 2 -- mean prev_30 CTR (%) by volume tier:")
signal2_table

Signal 2 -- mean prev_30 CTR (%) by volume tier:


,mean_ctr_pct,n
volume_tier,,
thin_lt500,0.159,9499
moderate_500_2000,0.222,6641
healthy_2000_5000,0.238,1684
high_5000plus,0.247,1094


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Encoding the exact flag from section 1: three eligibility gates (population), one condition
(evidence far below peers), one score (rank by stake). `expected_ctr_for_tier` is each row's
position tier's mean CTR **from this same `prev_30` slice**: not a future value, not a
warehouse-provided benchmark.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

os.makedirs("work/outputs", exist_ok=True)

# Population / eligibility -- gates, not weights
enough_impressions = df["prev_30_impressions"] >= 500                          # "enough impressions to judge"
not_brand_new       = df["content_age_days_at_decision"] >= 90                 # "not brand-new" (my documented choice)
not_in_cooldown      = df["days_since_last_update_at_decision"] >= 60          # "not optimized in the last two months" (deck's exact number)
eligible = enough_impressions & not_brand_new & not_in_cooldown

df_eligible = df[pd.Series(eligible).astype(bool)]
df["expected_ctr_for_tier"] = df_eligible.groupby("position_tier", observed=True)["prev_30_ctr"].transform("mean")
df["ctr_gap"] = df_eligible["expected_ctr_for_tier"] - df["prev_30_ctr"]        # positive = underperforming

# Condition -- "far below peers"
#far_below_peers = df["prev_30_ctr"] <= 0 * df["expected_ctr_for_tier"]        # documented choice, adjustable

far_below_peers = df["prev_30_ctr"] == 0

flagged = eligible & far_below_peers

df["score"] = np.where(flagged, df["prev_30_impressions"] * df["ctr_gap"], 0.0)   # rank by stake
df["reason_code"] = np.where(flagged, "CTR_BELOW_POSITION_PEERS", "")
df["action"] = np.where(flagged, "snippet_fix", "no_action")

queue = df.sort_values("score", ascending=False).reset_index(drop=True)

n_eligible = int(eligible.sum())
n_flagged = int(flagged.sum())
print(f"Ranked queue: {len(queue):,} rows total, {n_eligible:,} eligible, {n_flagged:,} flagged (score > 0).")

def precision_at_k(labels_arr, k):
    return labels_arr[:k].mean()

base_rate = df["is_declining"].mean()
print(f"\nBase rate (is_declining, whole slice): {base_rate:.3f}")
precisions = {}
for k in [10, 50, 100]:
    p = precision_at_k(queue["is_declining"].values, k)
    precisions[f"precision_at_{k}"] = float(p)
    print(f"precision@{k}: {p:.3f}   (vs base rate {base_rate:.3f})")

Ranked queue: 18,918 rows total, 8,165 eligible, 2,490 flagged (score > 0).

Base rate (is_declining, whole slice): 0.543
precision@10: 0.900   (vs base rate 0.543)
precision@50: 0.800   (vs base rate 0.543)
precision@100: 0.780   (vs base rate 0.543)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

For a CTR/position flag, the traps are things like a seasonal or branded-query dip in demand, a noisy position estimate, or the page being intentionally deprioritized (`is_published` / `is_deleted` in
`dim_content`).

`confidence` below is a simple, transparent heuristic, not a model output, based only on how much `prev_30` traffic backs the CTR estimate: a gap computed off 30,000 impressions is a lot more trustworthy than the same gap off 520.




In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

TOP_N = 10  # the card requires 10; change to 20 for the "always welcome" bonus

def confidence_note(row):
    if row["prev_30_impressions"] >= 5000:
        return "high (large prev_30 sample)"
    if row["prev_30_impressions"] >= 2000:
        return "medium"
    return "low (thin prev_30 sample for this position tier)"

top = queue.head(TOP_N).copy()
top["confidence"] = top.apply(confidence_note, axis=1)

review_cols = ["content_hash_id", "action", "reason_code", "position_tier", "prev_30_avg_position",
               "prev_30_ctr", "expected_ctr_for_tier", "ctr_gap", "prev_30_impressions",
               "content_age_days_at_decision", "days_since_last_update_at_decision", "confidence", "score"]
top[review_cols]

,content_hash_id,action,reason_code,position_tier,prev_30_avg_position,prev_30_ctr,expected_ctr_for_tier,ctr_gap,prev_30_impressions,content_age_days_at_decision,days_since_last_update_at_decision,confidence,score
0,content_b63094bd417908e9,snippet_fix,CTR_BELOW_POSITION_PEERS,page_2_3,40.132384,0.0,0.098199,0.098199,27563.0,114,65,high (large prev_30 sample),2706.663169
1,content_520e203a08cd69ee,snippet_fix,CTR_BELOW_POSITION_PEERS,top_3,0.139562,0.0,0.532067,0.532067,5074.0,218,65,high (large prev_30 sample),2699.707227
2,content_c0bfca959627c226,snippet_fix,CTR_BELOW_POSITION_PEERS,page_1,4.911741,0.0,0.258957,0.258957,9020.0,114,65,high (large prev_30 sample),2335.793183
3,content_41d18608d90de375,snippet_fix,CTR_BELOW_POSITION_PEERS,page_1,5.234949,0.0,0.258957,0.258957,8748.0,217,65,high (large prev_30 sample),2265.356848
4,content_62a498af66986229,snippet_fix,CTR_BELOW_POSITION_PEERS,top_3,2.839208,0.0,0.532067,0.532067,4060.0,218,65,medium,2160.191435
5,content_d397987113cb84a0,snippet_fix,CTR_BELOW_POSITION_PEERS,top_3,2.510213,0.0,0.532067,0.532067,3906.0,291,65,medium,2078.253139
6,content_76773f1111c808de,snippet_fix,CTR_BELOW_POSITION_PEERS,page_1,7.482591,0.0,0.258957,0.258957,7527.0,105,65,high (large prev_30 sample),1949.170210
7,content_17b9e821ceca190e,snippet_fix,CTR_BELOW_POSITION_PEERS,page_2_3,31.658238,0.0,0.098199,0.098199,18758.0,100,65,high (large prev_30 sample),1842.019654
8,content_ef16cd974ed64722,snippet_fix,CTR_BELOW_POSITION_PEERS,page_2_3,38.737472,0.0,0.098199,0.098199,17292.0,114,65,high (large prev_30 sample),1698.059700
9,content_f2a175ee4fa71a29,snippet_fix,CTR_BELOW_POSITION_PEERS,top_3,1.366214,0.0,0.532067,0.532067,3179.0,218,65,medium,1691.440535




1.   content_b63094bd417908e9 — action: snippet_fix. Why: prev_30_ctr is 0, expected_ctr_for_tier is 0.098199%, and prev_30_impressions is 27563. Would be wrong if: the gap is a seasonal or branded-query dip rather than a bad snippet, or this page was intentionally deprioritized (check is_published/is_deleted) Could be also that in low tiers like this it is normal to get no clicks.
2.   content_520e203a08cd69ee — action: snippet_fix. Why: prev_30_ctr is 0, expected_ctr_for_tier is 0.532067%, and prev_30_impressions is 5074. Would be wrong if: the gap is a seasonal or branded-query dip rather than a bad snippet, or this page was intentionally deprioritized (check is_published/is_deleted).
3. content_c0bfca959627c226 — action: snippet_fix. Why: prev_30_ctr is 0, expected_ctr_for_tier is 0.258957%, and prev_30_impressions is 9020. Would be wrong if: the gap is a seasonal or branded-query dip rather than a bad snippet, or this page was intentionally deprioritized (check is_published/is_deleted), or prev_30_avg_position moved a lot within the window (an average can hide real movement).
4. content_41d18608d90de375 — action: snippet_fix. Why: prev_30_ctr is 0, expected_ctr_for_tier is 0.258957%, and prev_30_impressions is 8748. Would be wrong if: the gap is a seasonal or branded-query dip rather than a bad snippet, or this page was intentionally deprioritized (check is_published/is_deleted) or prev_30_avg_position moved a lot within the window (an average can hide real movement).
5. content_62a498af66986229 — action: snippet_fix. Why: prev_30_ctr is 0, expected_ctr_for_tier is 0.532067%, and prev_30_impressions is 4060. Would be wrong if: the gap is a seasonal or branded-query dip rather than a bad snippet, or this page was intentionally deprioritized (check is_published/is_deleted). Also the number of impressions is not very large.
6. content_d397987113cb84a0 — action: snippet_fix. Why: prev_30_ctr is 0, expected_ctr_for_tier is 0.532067%, and prev_30_impressions is 3906. Would be wrong if: the gap is a seasonal or branded-query dip rather than a bad snippet, or this page was intentionally deprioritized (check is_published/is_deleted). Also the number of impressions is not very large.
7. content_76773f1111c808de — action: snippet_fix. Why: prev_30_ctr is 0, expected_ctr_for_tier is 0.258957%, and prev_30_impressions is 7527. Would be wrong if: the gap is a seasonal or branded-query dip rather than a bad snippet, or this page was intentionally deprioritized (check is_published/is_deleted), or prev_30_avg_position moved a lot within the window (an average can hide real movement).
8. content_17b9e821ceca190e — action: snippet_fix. Why: prev_30_ctr is 0, expected_ctr_for_tier is 0.098199%, and prev_30_impressions is 18758. Would be wrong if: the gap is a seasonal or branded-query dip rather than a bad snippet, or this page was intentionally deprioritized (check is_published/is_deleted), or prev_30_avg_position moved a lot within the window (an average can hide real movement).
9. content_ef16cd974ed64722 — action: snippet_fix. Why: prev_30_ctr is 0, expected_ctr_for_tier is 0.098199%, and prev_30_impressions is 17292. Would be wrong if: the gap is a seasonal or branded-query dip rather than a bad snippet, or this page was intentionally deprioritized (check is_published/is_deleted), or prev_30_avg_position moved a lot within the window (an average can hide real movement).
10. content_f2a175ee4fa71a29 — action: snippet_fix. Why: prev_30_ctr is 0, expected_ctr_for_tier is 0.532067%, and prev_30_impressions is 3179. Would be wrong if: the gap is a seasonal or branded-query dip rather than a bad snippet, or this page was intentionally deprioritized (check is_published/is_deleted), or prev_30_avg_position moved a lot within the window (an average can hide real movement). Also the number of impressions is not very large.





## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.